In [ ]:
! pip install finrl stable-baselines3 yfinance pandas-ta matplotlib seaborn gymnasium

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
import yfinance as yf 
from datetime import datetime, timedelta
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

In [2]:
import gymnasium as gym 
from gymnasium import spaces
from stable_baselines3 import PPO, A2C
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback

Initialization

In [3]:
TICKERS = [
    "RELIANCE.NS",    
    "TCS.NS",         
    "HDFCBANK.NS",    
    "INFY.NS",        
    "ICICIBANK.NS",   
]
TRAIN_START = "2018-01-01"
TRAIN_END   = "2022-12-31"
TEST_START  = "2023-01-01"
TEST_END    = "2024-12-31"
INITIAL_CAPITAL = 1000000   
MAX_SHARES_PER_STOCK = 50     

STT_RATE       = 0.001   # Securities Transaction Tax -> 0.1% on sell
BROKERAGE_RATE = 0.0003  # Flat brokerage->₹20 per order => apx 0.3%
GST_RATE       = 0.18    # GST
SLIPPAGE       = 0.001   # Market impact 

TRANSACTION_COST_RATE = STT_RATE + BROKERAGE_RATE * (1 + GST_RATE) + SLIPPAGE

In [11]:
ALGORITHM      = "PPO"        # ppo or a2c
TIMESTEPS      = 100000      # 500k
REWARD_SCALING = 1e-4         

# Technical Indicators 
MACD = True
RSI  = True
CCI  = True
ADX  = True

In [5]:
print(f" Stocks     : {TICKERS}")
print(f"Train      : {TRAIN_START} → {TRAIN_END}")
print(f"Test       : {TEST_START} → {TEST_END}")
print(f"Capital    : ₹{INITIAL_CAPITAL:,.0f}")
print(f"Txn Cost   : {TRANSACTION_COST_RATE*100:.3f}% per trade")
print(f"Algorithm  : {ALGORITHM}")
print(f"imesteps  : {TIMESTEPS:,}")

 Stocks     : ['RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'ICICIBANK.NS']
Train      : 2018-01-01 → 2022-12-31
Test       : 2023-01-01 → 2024-12-31
Capital    : ₹1,000,000
Txn Cost   : 0.235% per trade
Algorithm  : PPO
imesteps  : 100,000


Data Imports (NSE Market Data)

In [ ]:
def download_data(t, start, end):
    print(f"downloading data for {len(t)} stocks")
    datar=yf.download(t, start=start, end=end, group_by="ticker", auto_adjust=True, progress=False)
    frame=[]
    for ticker in t:
        try:
            df=datar[ticker][["Open", "High", "Low", "Close", "Volume"]].copy()
            df.columns = ["open", "high", "low", "close", "volume"]
            df["ticker"] = ticker
            df.index.name = "date"
            df = df.dropna()
            frame.append(df)
            print("passed")
        except Exception as e:
            print(f"{ticker}: {e}")
    combined = pd.concat(frame).reset_index()
    combined["date"] = pd.to_datetime(combined["date"])
    return combined


In [8]:
trainraw = download_data(TICKERS, TRAIN_START, TRAIN_END)
testraw  = download_data(TICKERS, TEST_START, TEST_END)

print(f"Train: {trainraw.shape}")
print(f"Test : {testraw.shape}")

downloading data for 5 stocks
passed
passed
passed
passed
passed
downloading data for 5 stocks
passed
passed
passed
passed
passed
Train: (6180, 7)
Test : (2450, 7)


In [10]:
trainraw.head()

,date,open,high,low,close,volume,ticker
0,2018-01-01,407.585208,407.585208,400.870911,401.864807,9453202,RELIANCE.NS
1,2018-01-02,403.300424,406.193768,400.384994,402.483215,9499419,RELIANCE.NS
2,2018-01-03,408.601166,409.042914,403.322483,404.095520,13507800,RELIANCE.NS
3,2018-01-04,405.575361,407.187693,404.493122,406.525085,9008932,RELIANCE.NS
4,2018-01-05,407.187664,409.440487,406.502971,407.828156,7441284,RELIANCE.NS


Feature Engineering

In [ ]:
def indicate(df):
    result=[]
    for ticker in df["ticker"].unique():
        stock = df[df["ticker"] == ticker].copy().sort_values("date")
        c = stock["close"]
        h = stock["high"]
        l = stock["low"]
        v = stock["volume"]

        if RSI:
            delta = c.diff()
            gain  = delta.clip(lower=0).rolling(14).mean()
            loss  = (-delta.clip(upper=0)).rolling(14).mean()
            rs    = gain / (loss + 1e-9)
            stock["rsi"] = 100 - (100 / (1 + rs))
        if MACD:
            ema12 = c.ewm(span=12, adjust=False).mean()
            ema26 = c.ewm(span=26, adjust=False).mean()
            stock["macd"]        = ema12 - ema26
            stock["macd_signal"] = stock["macd"].ewm(span=9, adjust=False).mean()
            stock["macd_hist"]   = stock["macd"] - stock["macd_signal"]
        
        if CCI:
            tp = (h + l + c) / 3
            ma = tp.rolling(20).mean()
            md = tp.rolling(20).apply(lambda x: np.mean(np.abs(x - x.mean())))
            stock["cci"] = (tp - ma) / (0.015 * md + 1e-9)
        
        if ADX:
            tr    = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
            atr   = tr.rolling(14).mean()
            plus  = (h - h.shift()).clip(lower=0)
            minus = (l.shift() - l).clip(lower=0)
            pdi   = 100 * plus.rolling(14).mean() / (atr + 1e-9)
            mdi   = 100 * minus.rolling(14).mean() / (atr + 1e-9)
            dx    = 100 * (pdi - mdi).abs() / (pdi + mdi + 1e-9)
            stock["adx"] = dx.rolling(14).mean()
        stock["volatility"] = c.pct_change().rolling(20).std()
        
        result.append(stock)
    
    out = pd.concat(result).dropna().reset_index(drop=True)
    print(f"Shape: {out.shape}")
    return out   

In [14]:
train = indicate(trainraw)
test  = indicate(testraw)

Shape: (6045, 14)
Shape: (2315, 14)


Environment init